# PEMS-SF 功能性日期分类 - LSTM-Kalman 训练笔记 (V1.1)
目标：不依赖日历标签，仅根据流量波形识别功能性日期（周一至周日），并缓解因调休导致的标签偏移。
数据：PEMS-SF，每日 144 槽占有率（5分钟粒度），已通过 Ward 聚类为 5 组（G1-G5）。本版本默认针对 G1（通勤双峰）训练。
说明：本 Notebook 仅保留训练所需的数据提取与模型代码，去除可视化输出。各步骤附交通流理论解释注释。

In [ ]:
# 基本依赖导入（模块化）
import os, pathlib, json, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
# SHAP 用于解释性分析（深度模型）
import shap
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT = pathlib.Path('.')
DATA_DIR = ROOT / 'data'
STAGE5_DIR = ROOT / 'Stage5-figure'
print('Device:', DEVICE)

In [ ]:
# 训练与解释的可复现性与确定性设置
torch.manual_seed(0); np.random.seed(0); random.seed(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## 数据加载与结构约束
- 交通流理论：通勤型（G1）在早晚高峰呈现双峰，工作日与周末差异显著，适合先行训练以稳定收敛。
- 输入：时序 $(144, 1)$；静态 5D（峰数、早峰、晚峰、工作日-周末差异、周末异常）。
- 约束：支持按聚类组选择子集（默认 G1）。

In [ ]:
# 读取 Stage5 生成的聚类与特征：为简化，这里从 station_weekday_time_profiles_long 构造 5D
# 若已存在预处理好的 5D 特征与分组文件，可直接读取以避免重复计算。
profiles_long_path = STAGE5_DIR / 'silhouette_results.json'  # 仅用来验证目录存在
if not STAGE5_DIR.exists():
    raise FileNotFoundError('缺少 Stage5-figure 目录，请先运行聚类与特征提取阶段（v3.5）。')

# 站点与标签日级数据来源：直接从原始 PEMS_train/PEMS_test 流式解析（与 v3.5 同逻辑）
train_path = ROOT / 'data' / 'PEMS_train'
test_path  = ROOT / 'data' / 'PEMS_test'
labels_train_path = ROOT / 'data' / 'PEMS_trainlabels'
labels_test_path  = ROOT / 'data' / 'PEMS_testlabels'
stations_list_path = ROOT / 'data' / 'stations_list'

def parse_day_matrix(line: str, expected_rows=963, expected_cols=144):
    line = line.strip()
    if line.startswith('[') and line.endswith(']'): line = line[1:-1]
    rows = [r.strip() for r in line.split(';') if r.strip()]
    data = []
    for r in rows:
        nums = [float(x) for x in r.split() if x.replace('.', '', 1).isdigit()]
        if nums: data.append(nums)
    if not data: return None
    arr = np.full((len(data), max(len(rr) for rr in data)), np.nan, dtype=float)
    for i, rr in enumerate(data): arr[i, :len(rr)] = rr
    return arr

def iter_day_matrices(path: pathlib.Path, limit=None):
    with path.open('r', encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            if limit is not None and i >= limit: break
            mat = parse_day_matrix(line)
            yield mat if mat is not None else None

def load_labels(path: pathlib.Path):
    txt = path.read_text(encoding='utf-8', errors='ignore').strip().strip('[]')
    return np.array([int(x) for x in txt.split() if x.isdigit()], dtype=int)

labels_train = load_labels(labels_train_path)
labels_test  = load_labels(labels_test_path)
station_ids = [int(x) for x in stations_list_path.read_text(encoding='utf-8', errors='ignore').strip().strip('[]').split() if x.isdigit()]
assert len(station_ids) == 963, '站点数量应为 963'
print('Train days:', len(labels_train), 'Test days:', len(labels_test))

In [ ]:
# 读取 G1..G5 分组索引并生成站点掩码（与 stations_list 顺序对齐）
groups_index_path = STAGE5_DIR / 'groups_g1_g5.json'
group_station_masks = {}
if groups_index_path.exists():
    groups_index = json.loads(groups_index_path.read_text(encoding='utf-8'))
    for g in ['g1','g2','g3','g4','g5']:
        selected_stations = set(groups_index.get(g, []))
        mask = np.array([sid in selected_stations for sid in station_ids], dtype=bool)
        group_station_masks[g] = mask
    print('已加载分组掩码，站点数:', {g: int(group_station_masks[g].sum()) for g in group_station_masks})
else:
    print('未找到分组 JSON，退回全体站点掩码。')
    group_station_masks = {g: np.ones(963, dtype=bool) for g in ['g1','g2','g3','g4','g5']}

## 5D Functional DNA 构造（无图输出）
交通流解释：5D 以形状特征概括拥堵动力学，峰值数量与时段强度反映通勤规律；工作日/周末差异捕捉制度性变化；周末异常用于标注非典型行为。

In [ ]:
from scipy.signal import find_peaks

# === 新增：简易 1D 卡尔曼滤波器 ===
def apply_kalman_filter(data: np.ndarray, n_iter=1) -> np.ndarray:
    """
    对 144 点的时间序列应用 1D 卡尔曼滤波进行去噪
    :param data: (144,) 原始观测数据
    :return: (144,) 平滑后的估计值
    """
    sz = (len(data),) 
    
    # 初始参数设定
    # Q: 过程噪声协方差 (假设真实交通流变化是平滑的) -> 越小越平滑
    # R: 观测噪声协方差 (假设传感器噪声较大) -> 越大越不信观测值
    Q = 1e-5 
    R = 0.01 
    
    xhat = np.zeros(sz)      # 后验估计
    P = np.zeros(sz)         # 后验误差协方差
    xhatminus = np.zeros(sz) # 先验估计
    Pminus = np.zeros(sz)    # 先验误差协方差
    K = np.zeros(sz)         # 卡尔曼增益

    # 初始化
    xhat[0] = data[0]
    P[0] = 1.0

    for k in range(1, len(data)):
        # 1. 预测阶段 (Time Update)
        xhatminus[k] = xhat[k-1] # 假设下一刻流量等于这一刻 (随机游走模型)
        Pminus[k] = P[k-1] + Q

        # 2. 更新阶段 (Measurement Update)
        K[k] = Pminus[k] / (Pminus[k] + R) # 计算卡尔曼增益
        xhatminus[k] = xhatminus[k]       # 这里的观测预测就是先验估计
        xhat[k] = xhatminus[k] + K[k] * (data[k] - xhatminus[k]) # 融合观测值
        P[k] = (1 - K[k]) * Pminus[k]

    return xhat.astype(np.float32)

def _clean_curve(curve: np.ndarray) -> np.ndarray:
    # 长度对齐与插值清洗，裁剪到 [0,1]
    curve = curve[:144]
    if curve.shape[0] < 144:
        curve = np.pad(curve, (0, 144 - curve.shape[0]), mode='constant', constant_values=np.nan)
    idx = np.arange(curve.size)
    mask = ~np.isnan(curve)
    if mask.any():
        curve = np.interp(idx, idx[mask], curve[mask])
    
    # === 修改：在插值后、归一化前应用卡尔曼滤波 ===
    # 这一步能有效去除毛刺，让 find_peaks 更准确，同时让 LSTM 输入更平滑
    curve = apply_kalman_filter(curve) 
    # ============================================

    curve = np.nan_to_num(curve, nan=0.0, posinf=1.0, neginf=0.0)
    curve = np.clip(curve, 0.0, 1.0)
    return curve.astype(np.float32)

def extract_5d_features(mat_963x144: np.ndarray):
    # 对每个站点曲线清洗后提取 5D 特征，避免 NaN 传播
    feats = []
    for i in range(mat_963x144.shape[0]):
        curve = _clean_curve(mat_963x144[i, :])
        peaks, _ = find_peaks(curve, prominence=0.03, width=2)
        n_peaks = min(len(peaks), 3) / 3.0
        denom = float(np.max(curve)) + 1e-8
        am_peak = float(np.max(curve[36:55])) / denom
        pm_peak = float(np.max(curve[96:115])) / denom
        wd_we_var = float(np.linalg.norm(curve[36:60] - curve[96:120]) / math.sqrt(24))
        wd_we_var = float(np.clip(wd_we_var, 0, 1))
        weekend_anom = float(np.clip(np.mean(curve[30:55]) / 0.5, 0, 1))
        feats.append([n_peaks, am_peak, pm_peak, wd_we_var, weekend_anom])
    return np.asarray(feats, dtype=np.float32)

## 数据集封装（按 Cluster 子集，如 G1）
交通流解释：在通勤型子群内训练可降低形状异质性，提高 LSTM 对双峰模式的表征稳定性。

In [ ]:
class PEMSFunctionalDataset(Dataset):
    def __init__(self, split_path: pathlib.Path, labels: np.ndarray, use_mask: np.ndarray | None = None):
        self.samples = []
        self._mask_warned = False
        for day_i, mat in enumerate(iter_day_matrices(split_path)):
            if day_i >= len(labels): break
            if mat is None or mat.shape[1] < 144: continue
            # 掩码长度必须与当日矩阵行一致
            if use_mask is not None:
                if use_mask.shape[0] == mat.shape[0]:
                    mat = mat[use_mask, :]
                elif not self._mask_warned:
                    print(f'警告：掩码长度({use_mask.shape[0]})≠当日矩阵行数({mat.shape[0]})，已跳过掩码。')
                    self._mask_warned = True
            feat5 = extract_5d_features(mat)
            for sidx in range(mat.shape[0]):
                seq = _clean_curve(mat[sidx, :144]).reshape(144, 1)
                f5 = feat5[sidx]
                # 丢弃任何非有限样本
                if not np.isfinite(seq).all() or not np.isfinite(f5).all():
                    continue
                orig_y = int(labels[day_i]) - 1
                # === 修改：映射到 4 类 (0:Mon, 1:Mid, 2:Fri, 3:Wknd) ===
                if orig_y == 0:   # Mon
                    y = 0
                elif orig_y in [1, 2, 3]: # Tue, Wed, Thu
                    y = 1
                elif orig_y == 4: # Fri
                    y = 2
                elif orig_y in [5, 6]: # Sat, Sun
                    y = 3
                else:
                    continue
                self.samples.append((seq.astype(np.float32), f5.astype(np.float32), y))
        self.n = len(self.samples)
    def __len__(self): return self.n
    def __getitem__(self, idx):
        seq, f5, y = self.samples[idx]
        return torch.from_numpy(seq), torch.from_numpy(f5), torch.tensor(y, dtype=torch.long)

## 多组数据准备与训练/验证拆分
- 使用 g1..g5 分别构建数据集。
- 对训练集按样本维度进行 80% 训练、20% 验证拆分。
- 训练阶段不使用测试集。

In [ ]:
# 为每个分组构建 Dataset (训练与测试)
group_datasets = {}
group_test_datasets = {}
for g, mask in group_station_masks.items():
    # 训练集源
    ds = PEMSFunctionalDataset(train_path, labels_train, mask)
    group_datasets[g] = ds
    # 测试集源
    ds_test = PEMSFunctionalDataset(test_path, labels_test, mask)
    group_test_datasets[g] = ds_test
    print(f'{g} 训练样本: {len(ds)}, 测试样本: {len(ds_test)}')

# 训练/验证拆分（随机），并封装测试集 DataLoader
group_loaders = {}
for g, ds in group_datasets.items():
    n_total = len(ds)
    n_train = int(n_total * 0.8)
    n_val = n_total - n_train
    train_subset, val_subset = random_split(ds, [n_train, n_val], generator=torch.Generator().manual_seed(0))
    train_loader = DataLoader(train_subset, batch_size=128, shuffle=True, num_workers=0)
    val_loader   = DataLoader(val_subset, batch_size=256, shuffle=False, num_workers=0)
    # 独立的测试集 Loader
    test_loader  = DataLoader(group_test_datasets[g], batch_size=256, shuffle=False, num_workers=0)
    group_loaders[g] = (train_loader, val_loader, test_loader)

## 模型定义：LSTM-Kalman Hybrid
交通流解释：LSTM 编码非线性拥堵动力学（高峰与回落）；融合 5D 静态形状特征提升对结构性流型的鲁棒性；卡尔曼后平滑抑制突发事件噪声。

In [ ]:
class LSTMKalmanClassifier(nn.Module):
    # 修改 num_classes 默认为 4
    def __init__(self, input_dim=1, hidden_dim=64, num_layers=2, feat5_dim=5, num_classes=4, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout)
        self.fusion_dim = hidden_dim + feat5_dim
        self.mlp = nn.Sequential(
            nn.Linear(self.fusion_dim, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, num_classes)  # <--- 关键点：这里输出了 7 个概率值（对应周一到周日）
        )
    def forward(self, seq_bt, feat5_bt):
        # seq_bt: (B,144,1), feat5_bt: (B,5)
        _, (h_n, _) = self.lstm(seq_bt)  # h_n: (num_layers, B, hidden)
        h = h_n[-1]  # 末层隐藏态 (B, hidden)
        x = torch.cat([h, feat5_bt], dim=1)
        logits = self.mlp(x)
        return logits

# 卡尔曼平滑（对类别概率的时间序列；这里按样本维度平滑批次的均值作为占位）
def kalman_smooth_probs(probs_t: np.ndarray, Q=1e-4, R=1e-3):
    # probs_t: (T, C) 时间序列（占位：本任务按日分类，T=1；若进行日序列推断可扩展）
    # 交通流解释：突发拥堵可能扰动短时预测，平滑可提升一致性。
    T, C = probs_t.shape
    x = probs_t.copy()
    # 简化：对每类独立 1D KF，当前占位返回原值（训练阶段不引入后处理影响损失）
    return x

## 多组训练循环与评估
- 对每个分组分别训练模型，使用验证集评估。
- 不再进行参数加权平均，而是基于分组路由的 MoE 评估逻辑：测试时，根据数据属于哪个组，调用该组对应的专家模型进行预测。

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# === 定义训练函数 ===
def train_one_group(train_loader, val_loader, epochs=50, lr=1e-4):
    model = LSTMKalmanClassifier().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4) # 降低LR，增加epochs

    # === 修改开始：计算并应用类别权重 ===
    # 粗略估计权重：样本越少，权重越大。
    # 假设均分是理想状态。Mid-Week约占3/7，Fri占1/7。
    # 权重建议：Mon: 1.0, Mid-Week: 0.4, Fri: 1.5 (加大惩罚), Weekend: 0.6
    # 这里的权重是为了强迫模型关注 Friday
    
    class_weights = torch.tensor([1.0, 1.0, 1.3, 1.1]).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)     # <--- 关键点：这是分类专用的损失函数，不是预测数值的 MSE
    
    best_acc = 0.0
    best_state = None
    
    for epoch in range(1, epochs+1):
        model.train()
        total_loss = 0
        for seq, f5, y in train_loader:
            seq, f5, y = seq.to(DEVICE).float(), f5.to(DEVICE).float(), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(seq, f5)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        # Val
        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for seq, f5, y in val_loader:
                seq, f5, y = seq.to(DEVICE).float(), f5.to(DEVICE).float(), y.to(DEVICE)
                out = model(seq, f5)
                preds = torch.argmax(out, dim=1)
                val_correct += (preds == y).sum().item()
                val_total += y.size(0)
        
        val_acc = val_correct / val_total if val_total > 0 else 0
        if val_acc > best_acc:
            best_acc = val_acc
            best_state = model.state_dict()
        
        if epoch % 10 == 0 or epoch == 1:
            print(f"  Epoch {epoch}: loss={total_loss/len(train_loader):.4f} val_acc={val_acc:.4f}")
        
    return best_state, best_acc

# === 1. 训练各组专家模型 ===
group_states = {}
print("开始训练各组专家模型 (MoE Strategy)...")

for g, (train_loader, val_loader, test_loader) in group_loaders.items():
    print(f'>>> 开始训练分组: {g}')
    # 增加 epochs 至 50，降低 lr 至 1e-4 以避免局部最优应
    state, val_acc = train_one_group(train_loader, val_loader, epochs=50, lr=1e-4)
    group_states[g] = state
    print(f"[{g}] 最佳验证准确率: {val_acc:.4f}")
    
    # 保存各组独立的专家模型
    if state is not None:
        save_path = ROOT / f'model_lstm_kalman_{g}_v112.pth'
        torch.save(state, save_path)
        print(f"已保存专家模型: {save_path}")

# === 2. 专家路由模式 (MoE) 评估 ===
# 不再进行参数平均，而是根据站点所属的 Group，路由到对应的专家模型进行预测
print("\n正在计算混淆矩阵 (专家路由模式)...")

all_y_true = []
all_y_pred = []

with torch.no_grad():
    for g, (_, _, test_loader) in group_loaders.items():
        # 加载该组对应的专家模型
        if g not in group_states or group_states[g] is None:
            print(f"警告: 组 {g} 未训练成功，跳过评估")
            continue
            
        expert_model = LSTMKalmanClassifier().to(DEVICE)
        expert_model.load_state_dict(group_states[g])
        expert_model.eval()
        
        for seq, f5, y in test_loader:
            seq = seq.to(DEVICE).float()
            f5 = f5.to(DEVICE).float()
            
            # 使用专家模型预测
            logits = expert_model(seq, f5)
            preds = torch.argmax(logits, dim=1)
            
            all_y_true.extend(y.cpu().numpy())
            all_y_pred.extend(preds.cpu().numpy())

# 生成混淆矩阵
cm = confusion_matrix(all_y_true, all_y_pred)

# 打印文本矩阵
print("\n=== 全网混淆矩阵 (MoE 路由预测 - 4 Class) ===")
print(cm)

# 计算每一类的准确率
class_acc = cm.diagonal() / cm.sum(axis=1)
days = ["Mon", "Mid-Week", "Fri", "Weekend"]
print("\n=== 各类识别准确率 ===")
for d, acc in zip(days, class_acc):
    print(f"{d}: {acc:.2%}")

## SHAP 解释性分析（DeepExplainer）
交通流解释：量化 5D 静态特征对某个预测结果的边际贡献，如早高峰强度对工作日识别的作用。

In [ ]:
class MLPForSHAP(nn.Module):
    # 修改 num_classes 默认为 4
    def __init__(self, fusion_dim=69, num_classes=4, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(fusion_dim, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, num_classes))
    
    def forward(self, x):
        return self.net(x)

# 确保推理模式，避免 Dropout 引入随机性导致 SHAP 可加性检查失败
# 修改：使用 G1 (通勤组) 的专家模型进行 SHAP 分析，因为 G1 最具代表性
mlp_shap = MLPForSHAP(fusion_dim=69).to(DEVICE)
mlp_shap.eval()

target_group = 'g1'
expert_model = LSTMKalmanClassifier().to(DEVICE)
expert_state_path = ROOT / f'model_lstm_kalman_{target_group}.pth'

if expert_state_path.exists():
    expert_model.load_state_dict(torch.load(expert_state_path, map_location=DEVICE))
    print(f'SHAP 分析：已加载 {target_group} 专家模型')
else:
    print(f'警告：未找到 {target_group} 模型，SHAP 将基于随机初始化权重。')

# 将专家模型的 MLP 权重复制到 SHAP MLP
mlp_layers = list(expert_model.mlp.children()); mlp_shap_layers = list(mlp_shap.net.children())
for i in range(len(mlp_layers)):
    if isinstance(mlp_layers[i], nn.Linear) and isinstance(mlp_shap_layers[i], nn.Linear):
        mlp_shap_layers[i].weight.data = mlp_layers[i].weight.data.detach().clone()
        mlp_shap_layers[i].bias.data = mlp_layers[i].bias.data.detach().clone()

# 背景样本（来自 g1 训练加载器）
bg = None
if target_group in group_loaders:
    # 使用目标组的数据作为背景
    train_loader = group_loaders[target_group][0]
    for i, (seq, f5, y) in enumerate(train_loader):
        if i >= 2: break
        with torch.no_grad():
            seq = seq.to(DEVICE).float(); f5 = f5.to(DEVICE).float()
            # 使用 expert_model 的 LSTM
            _, (h_n, _) = expert_model.lstm(seq)
            h = h_n[-1]
            fusion = torch.cat([h, f5], dim=1)
            bg = fusion if bg is None else torch.cat([bg, fusion], dim=0)
else:
    # Fallback
    for g, (train_loader, _) in group_loaders.items():
        break 

if bg is None:
    bg = torch.zeros((32, 69), device=DEVICE)
bg = torch.nan_to_num(bg, nan=0.0, posinf=1.0, neginf=0.0)

# 选择一个分组的小批次做解释
# 修改：从目标组取样
if target_group in group_loaders:
    seq_b, f5_b, y_b = next(iter(group_loaders[target_group][0]))
else:
    seq_b, f5_b, y_b = next(iter(next(iter(group_loaders.values()))[0]))

seq_b = seq_b.to(DEVICE).float(); f5_b = f5_b.to(DEVICE).float()
with torch.no_grad():
    _, (h_n, _) = expert_model.lstm(seq_b)
    h = h_n[-1]
    fusion_b = torch.cat([h, f5_b], dim=1)
fusion_b = torch.nan_to_num(fusion_b, nan=0.0, posinf=1.0, neginf=0.0)

try:
    explainer = shap.DeepExplainer(mlp_shap, bg)
    shap_values = explainer.shap_values(fusion_b, check_additivity=False)
except Exception as e:
    print('DeepExplainer 失败，回退 GradientExplainer:', str(e))
    explainer = shap.GradientExplainer(mlp_shap, bg)
    shap_values = explainer.shap_values(fusion_b)

# 仅输出 5D 特征维度的平均绝对贡献（无图）
feat_contrib = []
for c in range(4): # 修改 range(7) 为 range(4)
    sv = shap_values[c]  # (B, 69)
    f5_sv = np.abs(sv[:, -5:]).mean(axis=0)  # 取后 5 维
    feat_contrib.append(f5_sv.tolist())
print('5D 特征对各类的平均绝对贡献（样本批次）:', json.dumps(feat_contrib, ensure_ascii=False))

## 评估与阈值
(后续补充)